# Calculating Power-law scaling in fluctuations of information

In [20]:
import gc
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [21]:
CORPUS_NAME = 'null'

In [22]:
DATA_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [32]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [33]:
fs = [f for f in grab_all_files(DATA_PATH) if (not f.split('/')[-1].startswith('.'))]
len(fs)

1669

## Processing individual datasets

In [7]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [8]:
COEF_VAR = 'AVAR'

In [9]:
param_list = 'time_delta'

In [40]:
df_scale, df_regression, k_docs, pct = [], [], dict(), 1.

In [41]:
for f in tqdm(fs):

    # print(f.split('/')[-1])

    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()


    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['x_turn_id_'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df[param_list] = (df['y_turn_id'] - df['x_turn_id_']) + 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (~df[param_list].isna())
    ]

    for corpus in df['corpus'].unique():

        if CORPUS_NAME not in k_docs.keys():
            k_docs[CORPUS_NAME] = {
                True: 0,
                False: 0
            }

        sub = df.loc[df['corpus'].isin([corpus])]

        keep_data = sub['x_turn_id'].unique()

        keep_data = np.random.choice(keep_data, size=(int(np.ceil(pct*len(keep_data)),),), replace=False)

        sub = sub.loc[sub['x_turn_id'].isin(keep_data)]
        gc.collect()

        groups = sub.groupby(by=['conversation_id', 'x_speaker', 'self'])
        for labels, dfi in groups:

            if len(dfi):
                k_docs[CORPUS_NAME][labels[2]] += 1

                b,a = np.polyfit(np.log(dfi[param_list].sample(frac=1.).values), np.log(dfi['AVAR'].sample(frac=1.).values + 1e-9), 1)

                df_regression += [
                    {
                        'corpus': CORPUS_NAME,
                        'cid': labels[0],
                        'speaker': labels[1],
                        'self': labels[2],
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1669 [00:00<?, ?it/s]

9608


TypeError: tuple indices must be integers or slices, not list

In [11]:
# df_regression.isna().sum()

In [12]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [13]:
# if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         encoding='utf-8'
#     )
#
# else:
#     df_regression.to_csv(
#         os.path.join(OUTPUTS_PATH,'all.csv'),
#         index=False,
#         header=False,
#         encoding='utf-8',
#         mode='a'
#     )

## Corpus level analysis of results

In [11]:
split_by = ['corpus', 'self']

In [12]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [13]:
df0['t'] = df0['b'] / df0['se']

In [14]:
df0.head(n=20)

,corpus,self,b,std,k,se,t
0,null,False,-0.032884,0.918894,12604,0.008185,-4.017643
1,null,True,-0.062977,1.133411,11290,0.010667,-5.903969


In [15]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)